# # Phase 6 — Interpretability
# Rock Fragmentation Prediction from Blast Parameters
#
# Which blast-design and rock-property parameters drive the mean fragment
# size X50, and do the learned effects agree with established blasting
# physics? We use SHAP, permutation importance and partial-dependence (PDP)
# analysis on the best-performing models from Notebook 02.
#
# **Kaggle setup:** Add Input → `blast_master_110.csv` dataset (this
# notebook refits final models on the full 110-blast data with tuned
# hyper-parameters, so it does **not** strictly need Notebook 02's output —
# but if you saved it as a second dataset, you can load
# `tables/T4_results_P2.csv` to pick the top models automatically instead of
# hard-coding them below).

## Setup — sync with the GitHub repository
This project's [GitHub repo](https://github.com/mobashshir-ubn-zainuddin/Rock-fragmentation-analysis-post-blasting)
holds the dataset and, after each notebook pushes, every table, figure and
fitted model produced on Kaggle — this is what lets Notebooks 02 -> 03 -> 04
share results without manually re-creating a Kaggle Dataset each time.

**One-time setup (already done if `GITHUB_TOKEN` appears under *Add-ons ->
Secrets*):** create a GitHub fine-grained personal access token scoped only
to this repository with **Contents: Read and write** permission, add it as
a Kaggle secret named `GITHUB_TOKEN`, and attach that secret to this
notebook. **Internet must be ON** (Session options, right panel).

The token is read fresh from Kaggle Secrets for every git operation and is
never written to disk (passed as a one-off `git -c http.extraheader=...`),
so it cannot leak even if this notebook's output is later made public.

Because `sync_repo()` pulls the latest commits, this notebook automatically sees the outputs Notebook 02 already pushed.

In [ ]:
import base64
import subprocess
from pathlib import Path

GITHUB_REPO = "mobashshir-ubn-zainuddin/Rock-fragmentation-analysis-post-blasting"
REPO_DIR = Path("/kaggle/working/repo")


def _github_token():
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        print(f"GITHUB_TOKEN secret found (length {len(token)}).")
        return token
    except Exception as e:
        print(f"No GITHUB_TOKEN secret available ({type(e).__name__}: {e}); "
              "add one via Add-ons -> Secrets, attach it to this notebook, and "
              "turn Internet ON to enable GitHub sync. Falling back to the "
              "attached Kaggle dataset and /kaggle/working.")
        return None


def _auth_header(token):
    # GitHub's git-over-HTTPS endpoint expects HTTP Basic auth with the PAT as
    # the password (any non-empty username works; "x-access-token" is the
    # convention GitHub Actions itself uses) - NOT a bearer token, which is
    # only accepted by the REST/GraphQL API, not git's smart-HTTP protocol.
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    return f"http.extraheader=AUTHORIZATION: basic {basic}"


def _git(*args, token=None, cwd=None):
    """Run git; if a token is given it is passed as a one-off HTTP auth header
    (``-c http.extraheader=...``) so it is never written to .git/config or to
    any file under /kaggle/working - safe even if this notebook's output is
    later made public."""
    cmd = ["git"]
    if token:
        cmd += ["-c", _auth_header(token)]
    cmd += list(args)
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.returncode != 0 and r.stderr.strip():
        print(r.stderr.strip())
    return r


RESULTS_BRANCH = "kaggle-results"  # Kaggle never pushes to main, so it cannot clash with local work


def sync_repo():
    """Clone `main` (code + dataset) on first run in this session, then switch
    to the RESULTS_BRANCH, which holds every output pushed by earlier
    notebooks (created from main if it does not exist yet). Returns the local
    repo path, or None if no GITHUB_TOKEN secret is configured OR the sync
    actually failed (checked explicitly - git failures are not silently
    treated as success)."""
    token = _github_token()
    if token is None:
        return None
    import shutil
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    _git("config", "--global", "--add", "safe.directory", str(REPO_DIR))
    url = f"https://github.com/{GITHUB_REPO}.git"
    if (REPO_DIR / ".git").exists():
        r1 = _git("-C", str(REPO_DIR), "fetch", "origin", "main", token=token)
        r2 = _git("-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD", token=token)
        ok = r1.returncode == 0 and r2.returncode == 0
    else:
        if REPO_DIR.exists():
            # Leftover from an earlier failed sync or another notebook's
            # output - `git clone` refuses a non-empty, non-git directory.
            print(f"{REPO_DIR} exists but is not a git repository (leftover "
                  "from an earlier run) - removing it before cloning fresh.")
            shutil.rmtree(REPO_DIR)
        r = _git("clone", "--depth", "1", url, str(REPO_DIR), token=token)
        ok = r.returncode == 0 and (REPO_DIR / ".git").exists()
        if not ok:
            # A public repo can still be cloned anonymously if the token is
            # bad; reading works, but push_to_github() will still fail.
            print("Authenticated clone failed; retrying without a token "
                  "(works only if this repo is public) ...")
            if REPO_DIR.exists():
                shutil.rmtree(REPO_DIR)
            r = _git("clone", "--depth", "1", url, str(REPO_DIR))
            ok = r.returncode == 0 and (REPO_DIR / ".git").exists()
            if ok:
                print("Anonymous clone succeeded - this notebook can READ the "
                      "repo, but push_to_github() will fail until GITHUB_TOKEN "
                      "is valid and attached to THIS notebook.")
    if not ok:
        print("GitHub sync FAILED (see the git error above). Common causes: "
              "GITHUB_TOKEN not attached to THIS notebook (Add-ons -> Secrets "
              "-> tick the checkbox), Internet is OFF, the token expired, or "
              "its repo access/permissions don't cover this repo. Outputs "
              "will only be saved to /kaggle/working for this session.")
        return None
    _git("-C", str(REPO_DIR), "config", "user.email", "kaggle-bot@users.noreply.github.com")
    _git("-C", str(REPO_DIR), "config", "user.name", "Kaggle Notebook Bot")
    # Pick up results pushed by earlier notebooks, if the branch exists yet.
    rb = _git("-C", str(REPO_DIR), "fetch", "origin", RESULTS_BRANCH, token=token)
    if rb.returncode == 0:
        _git("-C", str(REPO_DIR), "checkout", "-B", RESULTS_BRANCH, "FETCH_HEAD")
        print(f"Checked out existing branch '{RESULTS_BRANCH}' (earlier results included).")
    else:
        _git("-C", str(REPO_DIR), "checkout", "-B", RESULTS_BRANCH)
        print(f"Branch '{RESULTS_BRANCH}' does not exist on GitHub yet (the "
              "error above is expected on the first run) - it will be created "
              "on the first push.")
    print(f"Repo synced at {REPO_DIR} (branch {RESULTS_BRANCH})")
    return REPO_DIR


def push_to_github(message, paths=("outputs",)):
    """Commit and push whatever changed under the given paths (relative to
    the repo root) to RESULTS_BRANCH. Safe to call repeatedly / after every
    protocol as a checkpoint, since a Kaggle session can be interrupted
    mid-run."""
    if REPO_DIR is None or not (REPO_DIR / ".git").exists():
        print("Repo not available (GitHub sync did not succeed earlier in "
              "this notebook) - skipping push. See the sync_repo() cell's "
              "output above for why.")
        return
    token = _github_token()
    if token is None:
        print("GITHUB_TOKEN no longer available - skipping push.")
        return
    _git("-C", str(REPO_DIR), "add", *paths)
    commit = _git("-C", str(REPO_DIR), "commit", "-m", message)
    if "nothing to commit" in (commit.stdout + commit.stderr):
        print("Nothing new to push.")
        return
    push = _git("-C", str(REPO_DIR), "push", "origin", f"HEAD:refs/heads/{RESULTS_BRANCH}", token=token)
    if push.returncode != 0:
        print("Push FAILED (see the git error above) - the commit above is "
              "still saved locally in /kaggle/working/repo and will be "
              "retried the next time push_to_github() is called. If the error "
              "says 403/permission denied, the token lacks Contents: Read and "
              "write on this repo.")
    else:
        print(f"Pushed to branch '{RESULTS_BRANCH}'.")


REPO_DIR = sync_repo()

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
CSV = None
for root in [Path("/kaggle/input"), REPO_DIR / "data" if REPO_DIR else None, Path("../data/processed")]:
    if root is not None and root.exists():
        found = list(root.rglob("blast_master_110.csv"))
        if found:
            CSV = found[0]
            break
assert CSV is not None, "blast_master_110.csv not found in /kaggle/input, the GitHub repo, or ../data/processed"
OUT = (REPO_DIR / "outputs") if REPO_DIR else (Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs"))
for sub in ["figures", "tables", "models"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV)
FEATURES = ["S_B", "H_B", "B_D", "T_B", "Pf", "XB", "E"]
TARGET = "X50"
XDF, y = df[FEATURES], df[TARGET].values
print(CSV, df.shape)

# ## 3.1 Re-import the model builders from Notebook 02
# (Paste `build()`, `space()`, `tune()` from Notebook 02 §2.3–2.4 here if
# running this notebook standalone on Kaggle — omitted for brevity; see
# `notebooks/02_baselines_and_ml_benchmark.ipynb` cells 5–9.)

In [ ]:
# --- model builders (identical to Notebook 02) ---
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from xgboost import XGBRegressor
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)


def logy(est):
    return TransformedTargetRegressor(regressor=est, func=np.log, inverse_func=np.exp)


def build(name, p):
    if name == "RF":
        return RandomForestRegressor(random_state=SEED, n_jobs=-1, **p)
    if name == "XGB":
        return XGBRegressor(random_state=SEED, n_jobs=4, verbosity=0, **p)
    if name == "SVR":
        return logy(make_pipeline(StandardScaler(), SVR(**p)))
    if name == "ANN":
        layers = tuple(p[f"n{i}"] for i in range(p["n_layers"]))
        return logy(make_pipeline(StandardScaler(), MLPRegressor(
            hidden_layer_sizes=layers, activation=p["act"], alpha=p["alpha"],
            solver="lbfgs", max_iter=5000, random_state=SEED)))
    raise ValueError(name)


def space(name, t):
    if name == "RF":
        return {"n_estimators": t.suggest_int("n_estimators", 200, 800, step=100),
                "max_depth": t.suggest_int("max_depth", 2, 12),
                "min_samples_leaf": t.suggest_int("min_samples_leaf", 1, 8),
                "max_features": t.suggest_float("max_features", 0.3, 1.0)}
    if name == "XGB":
        return {"n_estimators": t.suggest_int("n_estimators", 100, 1000, step=50),
                "learning_rate": t.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "max_depth": t.suggest_int("max_depth", 2, 6),
                "min_child_weight": t.suggest_float("min_child_weight", 1, 10),
                "subsample": t.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": t.suggest_float("colsample_bytree", 0.5, 1.0),
                "reg_lambda": t.suggest_float("reg_lambda", 1e-3, 10, log=True),
                "reg_alpha": t.suggest_float("reg_alpha", 1e-3, 10, log=True)}
    if name == "SVR":
        return {"C": t.suggest_float("C", 0.1, 1000, log=True),
                "epsilon": t.suggest_float("epsilon", 1e-3, 0.3, log=True),
                "gamma": t.suggest_float("gamma", 1e-3, 10, log=True)}
    if name == "ANN":
        n = t.suggest_int("n_layers", 1, 3)
        p = {"n_layers": n, "act": t.suggest_categorical("act", ["relu", "tanh"]),
             "alpha": t.suggest_float("alpha", 1e-5, 1.0, log=True)}
        for i in range(n):
            p[f"n{i}"] = t.suggest_int(f"n{i}", 4, 64, log=True)
        return p


DUP_GROUP = df.groupby(FEATURES, sort=False).ngroup().values
N_TRIALS = 40


def tune(name, Xtr, ytr, gtr):
    cv = GroupKFold(n_splits=5)

    def obj(t):
        est = build(name, space(name, t))
        return -cross_val_score(est, Xtr, ytr, groups=gtr, cv=cv,
                                 scoring="neg_root_mean_squared_error").mean()

    st = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    st.optimize(obj, n_trials=N_TRIALS, show_progress_bar=False)
    return space(name, optuna.trial.FixedTrial(st.best_params))

# ## 3.2 Fit final models on all 110 blasts
# Uses hyper-parameters retuned on the full dataset (5-fold grouped CV,
# duplicate-safe). These are the models used for every interpretability plot
# below — **not** the per-fold models from Notebook 02, which differ
# slightly fold to fold.

In [ ]:
final = {}
for name in ["XGB", "RF", "SVR", "ANN"]:
    p = tune(name, XDF.values, y, DUP_GROUP)
    final[name] = build(name, p).fit(XDF, y)
    print(name, "tuned params:", p)

import joblib

joblib.dump(final, OUT / "models/final_models.joblib")

# ## 3.3 SHAP — tree models (exact TreeExplainer)
# Figure 7: SHAP beeswarm summary for the best tree model (XGBoost).

In [ ]:
import matplotlib.pyplot as plt
import shap

sv = shap.TreeExplainer(final["XGB"])(XDF)
shap.plots.beeswarm(sv, show=False)
plt.tight_layout()
plt.savefig(OUT / "figures/F7_shap_beeswarm_XGB.png", dpi=300)
plt.show()

for f in ["Pf", "XB", "T_B", "E"]:
    shap.plots.scatter(sv[:, f], color=sv, show=False)
    plt.savefig(OUT / f"figures/shap_dep_{f}.png", dpi=300)
    plt.show()

# ## 3.4 SHAP — kernel explainer for non-tree models (SVR, ANN)
# `KernelExplainer` is model-agnostic but slower; a 20-point k-means
# background summary keeps it tractable at n=110.

In [ ]:
bg = shap.kmeans(XDF, 20)
for name in ["SVR", "ANN"]:
    ke = shap.KernelExplainer(final[name].predict, bg)
    vals = ke.shap_values(XDF, nsamples=300)
    shap.summary_plot(vals, XDF, show=False)
    plt.savefig(OUT / f"figures/shap_beeswarm_{name}.png", dpi=300)
    plt.show()

# ## 3.5 Permutation importance across all four models
# Computed with 30 repeats on the full dataset for the headline figure;
# for an unbiased per-model ranking also compute this on the held-out folds
# from Notebook 02 (`oof_P2.csv`) and compare — the two should broadly
# agree if the ranking is not an artefact of in-sample fitting.

In [ ]:
from sklearn.inspection import permutation_importance

imp = {}
for name, est in final.items():
    r = permutation_importance(est, XDF, y, n_repeats=30, random_state=SEED,
                                scoring="neg_root_mean_squared_error")
    imp[name] = pd.Series(r.importances_mean, index=FEATURES)
imp = pd.DataFrame(imp)
imp.to_csv(OUT / "tables/permutation_importance.csv")
print("Kendall tau between model rankings (consensus check):")
print(imp.rank().corr("kendall").round(2))
imp.round(4)

# ## 3.6 Partial dependence / ICE (Figure 8)

In [ ]:
from sklearn.inspection import PartialDependenceDisplay, partial_dependence

fig, ax = plt.subplots(figsize=(14, 7))
PartialDependenceDisplay.from_estimator(final["XGB"], XDF, FEATURES, kind="both",
                                         subsample=60, random_state=SEED, ax=ax)
plt.tight_layout()
plt.savefig(OUT / "figures/F8_pdp_ice_XGB.png", dpi=300)
plt.show()

# ## 3.7 Physics-consistency check
# Expected sign of the average trend from blasting theory (Cunningham 1987;
# Kulatilake et al. 2010): higher powder factor (`Pf`) → finer X50 (−);
# higher stemming ratio (`T_B`) and larger in-situ block size (`XB`) →
# coarser X50 (+); larger burden-to-diameter ratio (`B_D`) → coarser X50
# (+), though this ratio is nearly constant *within* a site (see Notebook
# 01 §1.5) so a mismatch here is a site-confounding signal, not necessarily
# a modelling failure — discuss both readings in the paper.

In [ ]:
expected = {"Pf": -1, "T_B": +1, "XB": +1, "B_D": +1}
consistency = {}
for name, est in final.items():
    signs = {}
    for f, e in expected.items():
        pdp = partial_dependence(est, XDF, [f], grid_resolution=20)
        slope = np.polyfit(pdp["grid_values"][0], pdp["average"][0], 1)[0]
        signs[f] = "OK" if np.sign(slope) == e else "VIOLATES"
    consistency[name] = signs
    print(name, signs)
pd.DataFrame(consistency).to_csv(OUT / "tables/physics_consistency.csv")

# ## Outputs of this notebook
# - `models/final_models.joblib` (used again in Notebook 04)
# - `figures/F7_shap_beeswarm_XGB.png`, `shap_dep_*.png`,
#   `shap_beeswarm_{SVR,ANN}.png`, `F8_pdp_ice_XGB.png`
# - `tables/permutation_importance.csv`, `physics_consistency.csv`

## Push results to GitHub
Pushes `models/final_models.joblib` and the SHAP/permutation-importance tables and figures, so Notebook 04 can load the fitted models.

In [ ]:
push_to_github("Notebook 03: interpretability outputs and fitted models", paths=["outputs"])